# baseline v3

이 베이스라인 코드는 `사전학습 모델 로드`, `배치 학습`, `파인튜닝`, `양자화`, `PEFT` 등이 적용된 버전입니다.

윈도우 데스크탑의 RTX 5060 ti GPU 환경에서 개발되었습니다.

# 환경 준비

개발 환경에 필요한 라이브러리 버전을 고정하고 최신 버전으로 라이브러리를 업데이트합니다.

- 아래 셀 실행
- ipykernel 설치
- 아래 셀 다시 실행 : 무한 로딩 시 restart
- hello 출력시 torch 설치

In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

# !cp /content/drive/MyDrive/data.zip /content/
# !unzip -q /content/data.zip -d /content/

In [2]:
# 압축이 잘 풀렸는지 확인
!ls -l /content/

total 1822192
-rw------- 1 root root 1862691773 Aug 29 09:17 data.zip
drwxr-xr-x 2 root root     135168 Aug 28 09:44 dev
-rw-r--r-- 1 root root     707055 Aug 28 09:43 dev.csv
drwx------ 5 root root       4096 Aug 29 09:16 drive
-rw-r--r-- 1 root root     162496 Aug 29 11:36 probability_1500.npy
drwxr-xr-x 2 root root       4096 Aug 29 09:36 qwen_27b_lora_max
drwxr-xr-x 2 root root       4096 Aug 29 10:22 qwen_8b_1500
drwxr-xr-x 1 root root       4096 Aug 24 13:28 sample_data
-rw-r--r-- 1 root root      81195 Aug 28 09:44 sample_submission.csv
-rw-r--r-- 1 root root      81194 Aug 29 11:36 submission_1500.csv
drwxr-xr-x 2 root root     176128 Aug 28 09:45 test
-rw-r--r-- 1 root root     830494 Aug 28 09:44 test.csv
drwxr-xr-x 2 root root     176128 Aug 28 09:46 train
-rw-r--r-- 1 root root     852411 Aug 28 09:45 train.csv


In [3]:
import sys
print(sys.executable)

/usr/bin/python3


In [4]:
!{sys.executable} -m pip uninstall -y torch torchvision torchaudio

Found existing installation: torch 2.11.0+cu128
Uninstalling torch-2.11.0+cu128:
  Successfully uninstalled torch-2.11.0+cu128
Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128


In [5]:
import sys

# 1. 기존 PyTorch 삭제 및 안정성이 검증된 버전으로 재설치
!{sys.executable} -m pip uninstall -y torch torchvision torchaudio
!{sys.executable} -m pip install torch==2.11.0 torchvision==0.26.0 torchaudio==2.11.0 --index-url https://download.pytorch.org/whl/cu128

# 2. 설치 확인
import torch
print("Torch version:", torch.__version__)
print("GPU Device:", torch.cuda.get_device_name())

# 3. 필수 라이브러리 설치 및 에러 유발 패키지(torchao) 삭제
!pip -q install "transformers>=4.43.2,<5.0.0" "accelerate>=0.34.2" "peft>=0.13.2" "bitsandbytes>=0.43.3" datasets pillow pandas --upgrade
!pip uninstall -y -q torchao

Looking in indexes: https://download.pytorch.org/whl/cu128
  Using cached https://download-r2.pytorch.org/whl/cu128/torch-2.11.0%2Bcu128-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (29 kB)
  Using cached https://download-r2.pytorch.org/whl/cu128/torchvision-0.26.0%2Bcu128-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached https://download-r2.pytorch.org/whl/cu128/torchaudio-2.11.0%2Bcu128-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (6.9 kB)
Using cached https://download-r2.pytorch.org/whl/cu128/torch-2.11.0%2Bcu128-cp313-cp313-manylinux_2_28_x86_64.whl (820.3 MB)
Using cached https://download-r2.pytorch.org/whl/cu128/torchvision-0.26.0%2Bcu128-cp313-cp313-manylinux_2_28_x86_64.whl (8.1 MB)
Using cached https://download-r2.pytorch.org/whl/cu128/torchaudio-2.11.0%2Bcu128-cp313-cp313-manylinux_2_28_x86_64.whl (1.7 MB)
Torch version: 2.11.0+cu128
GPU Device: NVIDIA A100-SXM4-40GB


In [6]:
print('hello123')

hello123


# 데이터 준비

개발에 필요한 데이터를 준비합니다.

- train.csv, train 폴더
- test.csv, test 폴더
- sample_submission.csv

데이터를 압축 해제하는데 몇 분 정도의 시간이 소요됩니다.

# 라이브러리, 데이터, 설정

In [7]:
import os, random, math
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
from typing import Any
from transformers import (
    AutoModelForVision2Seq,
    AutoProcessor,
    BitsAndBytesConfig,
    get_linear_schedule_with_warmup
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from tqdm.auto import tqdm

Image.MAX_IMAGE_PIXELS = None
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# --- 최종 밤샘 구동 세팅 ---
# MODEL_ID = "Qwen/Qwen3.8-27B"  # 8B를 쓴다면 "Qwen/Qwen3-VL-8B-Instruct"로 변경
MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"
IMAGE_SIZE = 1024              # 고해상도 (VQA 성능 핵심)
MAX_NEW_TOKENS = 8
SEED = 42

random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

train_df = pd.read_csv("train.csv")
test_df  = pd.read_csv("test.csv")

# 전체 데이터 학습(5073개)을 진행하려면 아래 줄을 주석 처리하세요.
# 만약 1500장 서브셋 학습을 원하시면 주석을 해제하세요.
# train_df = train_df.sample(n=1500, random_state=SEED).reset_index(drop=True)

Device: cuda


# 모델, Processor

7.5GB 정도의 모델 다운로드가 진행됩니다. 10~20분 정도가 소요됩니다.

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - LoRA 구현 : LoraConfig()

In [8]:
# # 양자화 - 27b
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_use_double_quant=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.bfloat16,
# )

# 프로세서
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=256*256,
    max_pixels=IMAGE_SIZE*IMAGE_SIZE,
    trust_remote_code=True,
)
processor.tokenizer.padding_side = "left"

# 사전학습 모델
# 27b 모델 기준 + 양자화
# base_model = AutoModelForVision2Seq.from_pretrained(
#     MODEL_ID,
#     quantization_config=bnb_config,
#     device_map="cuda",
#     trust_remote_code=True,
# )
# 8b + 양자화 X
base_model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="cuda",
    attn_implementation="sdpa",
    trust_remote_code=True,
)


# 양자화 모델로 로드
# base_model = prepare_model_for_kbit_training(base_model)
base_model.gradient_checkpointing_enable()

# LoRA 세팅
lora_config = LoraConfig(
    # r=16,            # 27b
    r = 32,            # 8b
    # lora_alpha=32,   # 27b
    lora_alpha=64,     # 8b
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    task_type="CAUSAL_LM",
)

# PEFT 모델 생성
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

trainable params: 87,293,952 || all params: 8,854,417,648 || trainable%: 0.9859


# 프롬프트 템플릿

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - 프롬프트 템플릿 : convert_to_chatml(), formatting_prompts_func()

In [9]:
# 모델 지시사항 (한국어)
SYSTEM_INSTRUCT = (
    "당신은 시각적 질의응답(VQA)을 완벽하게 수행하는 유능한 인공지능입니다. "
    "주어진 이미지와 질문을 분석한 뒤, 반드시 a, b, c, d 중 하나의 소문자 알파벳으로만 답변하세요."
)

# 프롬프트 (한국어)
def build_mc_prompt(question, a, b, c, d):
    return f"질문: {question}\n(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n정답:"

# Custom Dataset, Collator

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - TensorDataset()

    챕터 5-2 데이터 생성 및 파인튜닝 (향후 학습 분량)
    - IntentDataset()

In [10]:
class VQAMCDataset(Dataset):
    def __init__(self, df, processor, train=True):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.train = train

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["path"]).convert("RGB")
        q = str(row["question"])
        options = [str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])]

        if self.train:
            ans_char = str(row["answer"]).strip().lower()
            ans_idx = ['a', 'b', 'c', 'd'].index(ans_char)
            correct_text = options[ans_idx]
            random.shuffle(options)
            final_answer = ['a', 'b', 'c', 'd'][options.index(correct_text)]
        else:
            final_answer = None

        user_text = build_mc_prompt(q, options[0], options[1], options[2], options[3])
        messages = [
            {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
            {"role":"user","content":[{"type":"image","image":img}, {"type":"text","text":user_text}]}
        ]

        if self.train:
            messages.append({"role":"assistant","content":[{"type":"text","text":final_answer}]})

        return {"messages": messages, "image": img, "prompt_only_messages": messages[:-1] if self.train else messages}

@dataclass
class DataCollator:
    processor: Any
    train: bool = True

    def __call__(self, batch):
        texts, prompt_texts, images = [], [], []
        for sample in batch:
            texts.append(self.processor.apply_chat_template(sample["messages"], tokenize=False, add_generation_prompt=False))
            prompt_texts.append(self.processor.apply_chat_template(sample["prompt_only_messages"], tokenize=False, add_generation_prompt=True))
            images.append(sample["image"])

        enc = self.processor(text=texts, images=images, padding=True, return_tensors="pt")
        prompt_enc = self.processor(text=prompt_texts, images=images, padding=True, return_tensors="pt")

        if self.train:
            labels = enc["input_ids"].clone()
            labels[labels == self.processor.tokenizer.pad_token_id] = -100
            for i in range(len(batch)):
                prompt_len = (prompt_enc["input_ids"][i] != self.processor.tokenizer.pad_token_id).sum()
                labels[i, :prompt_len] = -100
            enc["labels"] = labels
        return enc

# DataLoader

#### 실습 참고 내용

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 데이터로더 정의 : DataLoader()

In [11]:
# 검증용 데이터 분리
split = int(len(train_df)*0.9)

train_subset, valid_subset = train_df.iloc[:split], train_df.iloc[split:]

# VQAMCDataset 형태로 변환
train_ds = VQAMCDataset(train_subset, processor, train=True)
valid_ds = VQAMCDataset(valid_subset, processor, train=True)


# 데이터로더
train_loader = DataLoader(
    train_ds,
    batch_size=1,
    shuffle=True,
    collate_fn=DataCollator(processor, True),
    num_workers=0)
valid_loader = DataLoader(
    valid_ds,
    batch_size=1,
    shuffle=False,
    collate_fn=DataCollator(processor, True),
    num_workers=0)

# fine-tuning

- 200개만 학습 : 10~20분 소요

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - 모델 정의 : SimpleMLP(), SequentialMLP()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()

In [12]:
model = model.to(device)
# GRAD_ACCUM = 16  # 27b, 유효 배치 사이즈 16 (안정적인 학습)
GRAD_ACCUM = 8  # 업데이트 주기를 적절히 당김

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
num_training_steps = 1 * math.ceil(len(train_loader) / GRAD_ACCUM)
scheduler = get_linear_schedule_with_warmup(optimizer, int(num_training_steps*0.03), num_training_steps)

# 최신 문법의 scaler 적용
scaler = torch.amp.GradScaler('cuda', enabled=True)

for epoch in range(1): # 무조건 1에폭 고정
    running = 0.0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1} [train]", unit="batch")

    for step, batch in enumerate(progress_bar, start=1):
        batch = {k:v.to(device) for k,v in batch.items()}
        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            outputs = model(**batch)
            loss = outputs.loss / GRAD_ACCUM

        scaler.scale(loss).backward()
        running += loss.item()

        if step % GRAD_ACCUM == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            progress_bar.set_postfix({"loss": f"{running / GRAD_ACCUM:.3f}"})
            running = 0.0

    model.eval()
    val_loss, val_steps = 0.0, 0
    with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.bfloat16):
        for vb in tqdm(valid_loader, desc=f"Epoch {epoch+1} [valid]", unit="batch"):
            vb = {k:v.to(device) for k,v in vb.items()}
            val_loss += model(**vb).loss.item()
            val_steps += 1
    print(f"[Epoch {epoch+1}] valid loss {val_loss/val_steps:.4f}")
    model.train()

# 모델 저장
SAVE_DIR = "/content/qwen_8b_all"
model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)
print("Saved:", SAVE_DIR)


Epoch 1 [train]:   0%|          | 0/4565 [00:00<?, ?batch/s]

Epoch 1 [valid]:   0%|          | 0/508 [00:00<?, ?batch/s]

[Epoch 1] valid loss 0.0713
Saved: /content/qwen_8b_all


# inference

30분~1시간 소요

#### 실습 참고 내용

    챕터4-1 RAG 기반 Customer Service AI 에이전트 개발
    - 데이터 파서 : langchain_core.output_parsers(), StrOutputParser()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()

In [13]:
model.eval()
candidate_tokens = ['a', 'b', 'c', 'd']
candidate_ids = [processor.tokenizer.encode(t, add_special_tokens=False)[0] for t in candidate_tokens]

ROTATIONS = 4
# BATCH_SIZE = 1     # 27B+1024px 모델은 추론도 batch 1이 안전합니다
BATCH_SIZE = 4  # 8b

n_samples = len(test_df)

P_total = np.zeros((n_samples, 4))

for rot in range(ROTATIONS):
    for s in tqdm(range(0, n_samples, BATCH_SIZE), desc=f"Inference [TTA Rot {rot}]", unit="batch"):
        chunk = test_df.iloc[s:s+BATCH_SIZE]
        texts, images, perms = [], [], []

        for _, row in chunk.iterrows():
            img = Image.open(row["path"]).convert("RGB")
            q = str(row["question"])
            base_opts = [str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])]

            # 선지 위치 순환
            perm = [(i + rot) % 4 for i in range(4)]
            user_text = build_mc_prompt(q, base_opts[perm[0]], base_opts[perm[1]], base_opts[perm[2]], base_opts[perm[3]])

            messages = [
                {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
                {"role":"user","content":[{"type":"image","image":img}, {"type":"text","text":user_text}]}
            ]
            texts.append(processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
            images.append(img)
            perms.append(perm)

        inputs = processor(text=texts, images=images, padding=True, return_tensors="pt").to(device)

        with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.bfloat16):
            outputs = model(**inputs)
            logits = outputs.logits[:, -1, :].float()
            lp = torch.log_softmax(logits[:, candidate_ids], dim=-1).cpu().numpy()

        for k, perm in enumerate(perms):
            for shown_i, orig_i in enumerate(perm):
                P_total[s+k, orig_i] += lp[k, shown_i]

        # VRAM 메모리 파편화 방지
        del inputs, outputs
        torch.cuda.empty_cache()

# 평균 확률로 정답 도출
P_total /= ROTATIONS
final_preds = [candidate_tokens[idx] for idx in P_total.argmax(axis=1)]

# 확률값을 .npy 파일로 저장 (앙상블)
np.save("probability_all.npy", P_total)
print("확률값이 probability_all.npy 로 저장되었습니다.")

# 제출 파일 저장 (안전한 최상위 경로)
submission = pd.DataFrame({"id": test_df["id"], "answer": final_preds})
submission.to_csv("/content/submission_all.csv", index=False)
print("성공적으로 저장되었습니다: /content/submission_all.csv")

Inference [TTA Rot 0]:   0%|          | 0/1269 [00:00<?, ?batch/s]

Inference [TTA Rot 1]:   0%|          | 0/1269 [00:00<?, ?batch/s]

Inference [TTA Rot 2]:   0%|          | 0/1269 [00:00<?, ?batch/s]

Inference [TTA Rot 3]:   0%|          | 0/1269 [00:00<?, ?batch/s]

확률값이 probability_all.npy 로 저장되었습니다.
성공적으로 저장되었습니다: /content/submission_all.csv


In [14]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ==========================================
# [현재 로드된 모델용] dev.csv 추론 및 .npy 저장
# ==========================================
model.eval()
candidate_tokens = ['a', 'b', 'c', 'd']
candidate_ids = [processor.tokenizer.encode(t, add_special_tokens=False)[0] for t in candidate_tokens]

ROTATIONS = 4
BATCH_SIZE = 4
dev_df = pd.read_csv("dev.csv") # test_df 대신 dev_df 사용
n_samples_dev = len(dev_df)

P_total_dev = np.zeros((n_samples_dev, 4))

for rot in range(ROTATIONS):
    for s in tqdm(range(0, n_samples_dev, BATCH_SIZE), desc=f"Dev Inference [TTA Rot {rot}]", unit="batch"):
        chunk = dev_df.iloc[s:s+BATCH_SIZE]
        texts, images, perms = [], [], []

        for _, row in chunk.iterrows():
            img = Image.open(row["path"]).convert("RGB")
            q = str(row["question"])
            base_opts = [str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])]
            perm = [(i + rot) % 4 for i in range(4)]
            user_text = build_mc_prompt(q, base_opts[perm[0]], base_opts[perm[1]], base_opts[perm[2]], base_opts[perm[3]])

            messages = [
                {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
                {"role":"user","content":[{"type":"image","image":img}, {"type":"text","text":user_text}]}
            ]
            texts.append(processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
            images.append(img)
            perms.append(perm)

        inputs = processor(text=texts, images=images, padding=True, return_tensors="pt").to(device)

        with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.bfloat16):
            outputs = model(**inputs)
            logits = outputs.logits[:, -1, :].float()
            lp = torch.log_softmax(logits[:, candidate_ids], dim=-1).cpu().numpy()

        for k, perm in enumerate(perms):
            for shown_i, orig_i in enumerate(perm):
                P_total_dev[s+k, orig_i] += lp[k, shown_i]

        del inputs, outputs
        torch.cuda.empty_cache()

P_total_dev /= ROTATIONS

# Dev 확률값 저장 (All 모델)
np.save("dev_prob_all.npy", P_total_dev)
print("🎉 성공: dev_prob_all.npy 가 저장되었습니다.")

Dev Inference [TTA Rot 0]:   0%|          | 0/1104 [00:00<?, ?batch/s]

Dev Inference [TTA Rot 1]:   0%|          | 0/1104 [00:00<?, ?batch/s]

In [ ]:
from peft import PeftModel

# 1. 1500장 모델 가중치 덮어씌우기 (학습 불필요)
# base_model은 이미 8B로 로드되어 있으므로, PEFT 어댑터만 1500버전으로 갈아끼웁니다.
model = PeftModel.from_pretrained(base_model, "/content/qwen_8b_1500")
model = model.to(device)
model.eval()

P_total_dev_1500 = np.zeros((n_samples_dev, 4))

# 2. 추론 루프 (Step 1과 완전히 동일)
for rot in range(ROTATIONS):
    for s in tqdm(range(0, n_samples_dev, BATCH_SIZE), desc=f"Dev Inference 1500 [TTA Rot {rot}]", unit="batch"):
        chunk = dev_df.iloc[s:s+BATCH_SIZE]
        texts, images, perms = [], [], []

        for _, row in chunk.iterrows():
            img = Image.open(row["path"]).convert("RGB")
            q = str(row["question"])
            base_opts = [str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])]
            perm = [(i + rot) % 4 for i in range(4)]
            user_text = build_mc_prompt(q, base_opts[perm[0]], base_opts[perm[1]], base_opts[perm[2]], base_opts[perm[3]])

            messages = [
                {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
                {"role":"user","content":[{"type":"image","image":img}, {"type":"text","text":user_text}]}
            ]
            texts.append(processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
            images.append(img)
            perms.append(perm)

        inputs = processor(text=texts, images=images, padding=True, return_tensors="pt").to(device)

        with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.bfloat16):
            outputs = model(**inputs)
            logits = outputs.logits[:, -1, :].float()
            lp = torch.log_softmax(logits[:, candidate_ids], dim=-1).cpu().numpy()

        for k, perm in enumerate(perms):
            for shown_i, orig_i in enumerate(perm):
                P_total_dev_1500[s+k, orig_i] += lp[k, shown_i]

        del inputs, outputs
        torch.cuda.empty_cache()

P_total_dev_1500 /= ROTATIONS

# Dev 확률값 저장 (1500 모델)
np.save("dev_prob_1500.npy", P_total_dev_1500)
print("🎉 성공: dev_prob_1500.npy 가 저장되었습니다.")

In [14]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# ==========================================
# 1. 파일 로드 및 준비
# ==========================================
# (주의: 파일 경로는 실제 저장하신 경로에 맞게 수정하세요)
try:
    dev_df = pd.read_csv("dev.csv")
    test_df = pd.read_csv("test.csv")

    # Dev 확률 로드 (반드시 확보해야 함)
    dev_p_all = np.load("dev_prob_all.npy")
    dev_p_1500 = np.load("dev_prob_1500.npy")

    # Test 확률 로드 (이미 확보 완료)
    test_p_all = np.load("probability_all.npy")
    test_p_1500 = np.load("probability_1500.npy")
except FileNotFoundError as e:
    print(f"🚨 에러: {e}")
    print("먼저 dev.csv에 대한 모델별 추론(.npy)을 진행해 주세요!")
    raise

# 정답 알파벳 매핑
classes = np.array(['a', 'b', 'c', 'd'])

# ==========================================
# 2. Dev 정답(Gold Label) 다수결 추출
# ==========================================
ans_cols = ['answer1', 'answer2', 'answer3', 'answer4', 'answer5']
# 5명 중 가장 많이 나온 알파벳을 정답으로 간주
dev_df['gold'] = dev_df[ans_cols].mode(axis=1, dropna=True)[0]
y_true = dev_df['gold'].values

# ==========================================
# 3. 오답 교차 검증 (Error Complementarity)
# ==========================================
pred_all = classes[dev_p_all.argmax(axis=1)]
pred_1500 = classes[dev_p_1500.argmax(axis=1)]

acc_all = np.mean(pred_all == y_true)
acc_1500 = np.mean(pred_1500 == y_true)

both_correct = np.sum((pred_all == y_true) & (pred_1500 == y_true))
all_c_1500_w = np.sum((pred_all == y_true) & (pred_1500 != y_true))
all_w_1500_c = np.sum((pred_all != y_true) & (pred_1500 == y_true))
both_wrong = np.sum((pred_all != y_true) & (pred_1500 != y_true))

print("-" * 40)
print(f"📊 [단일 모델 Dev 정확도]")
print(f"전체(All) 모델 : {acc_all:.4f}")
print(f"1500장 모델   : {acc_1500:.4f}")
print("-" * 40)
print(f"🔍 [오답 교차 분석 (Complementarity)]")
print(f"둘 다 정답              : {both_correct}")
print(f"All 정답 / 1500 오답    : {all_c_1500_w}")
print(f"All 오답 / 1500 정답    : {all_w_1500_c}  <-- 💡 이 숫자가 클수록 앙상블 대박!")
print(f"둘 다 오답              : {both_wrong}  <-- 🚨 2차 Specialist의 타겟!")
print("-" * 40)

# ==========================================
# 4. 황금 가중치 자동 탐색 (Weight Sweep)
# ==========================================
best_ari_w, best_ari_acc = 0, 0
best_geo_w, best_geo_acc = 0, 0

print("\n🚀 [0.00 ~ 1.00 앙상블 황금 비율 탐색 중...]")
for w in np.arange(0.0, 1.01, 0.01):
    # 1) 산술 평균 (Arithmetic Mean)
    p_ari = w * dev_p_all + (1 - w) * dev_p_1500
    acc_ari = np.mean(classes[p_ari.argmax(axis=1)] == y_true)
    if acc_ari > best_ari_acc:
        best_ari_acc = acc_ari
        best_ari_w = w

    # 2) 기하 평균 (Geometric Mean) - log(0) 방지를 위해 clip 적용
    p_geo = (np.clip(dev_p_all, 1e-12, 1) ** w) * (np.clip(dev_p_1500, 1e-12, 1) ** (1 - w))
    acc_geo = np.mean(classes[p_geo.argmax(axis=1)] == y_true)
    if acc_geo > best_geo_acc:
        best_geo_acc = acc_geo
        best_geo_w = w

print(f"🏆 최고 산술평균(Arith) 가중치: All({best_ari_w:.2f}) / 1500({1-best_ari_w:.2f}) -> 정확도: {best_ari_acc:.4f}")
print(f"🏆 최고 기하평균(Geom) 가중치: All({best_geo_w:.2f}) / 1500({1-best_geo_w:.2f}) -> 정확도: {best_geo_acc:.4f}")

# ==========================================
# 5. 최적 방식 선택 및 Test 데이터에 적용
# ==========================================
if best_ari_acc >= best_geo_acc:
    print(f"\n✅ 산술평균 방식 채택! 가중치 {best_ari_w:.2f}을 Test에 적용합니다.")
    final_p_test = best_ari_w * test_p_all + (1 - best_ari_w) * test_p_1500
else:
    print(f"\n✅ 기하평균 방식 채택! 가중치 {best_geo_w:.2f}을 Test에 적용합니다.")
    final_p_test = (np.clip(test_p_all, 1e-12, 1) ** best_geo_w) * (np.clip(test_p_1500, 1e-12, 1) ** (1 - best_geo_w))

# 최종 결과 저장
final_preds = classes[final_p_test.argmax(axis=1)]
submission = pd.DataFrame({"id": test_df["id"], "answer": final_preds})
submission.to_csv("final_ensemble_submission.csv", index=False)
print("🎉 성공적으로 저장되었습니다: final_ensemble_submission.csv")